# Classical Models

In [34]:
import pandas as pd
import numpy as np

from imblearn.over_sampling import SMOTE

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier

## Data setup

In [35]:
clinical = pd.read_csv('../data/Demo_data.csv')

X = clinical.drop(columns = ['Patient ID', 'Steatosis stage', 'Fibroscan S', 'Fibroscan F'])
Y = clinical[['Steatosis stage']]

encoder = OneHotEncoder(drop = 'if_binary', sparse_output = False)
X[['sex']] = encoder.fit_transform(X[['sex']])

In [36]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size = 0.25, random_state = 1234)

## Train with 4 classes (S0, S1, S2, S3)

In [37]:
def trainRandomForest(x_train, x_test, y_train, y_test):
    rf = RandomForestClassifier(class_weight = 'balanced')

    param_grid = {
        'n_estimators': [20, 50, 100, 125, 150, 175, 200],
        'max_depth': [None, 5, 10, 20],
        'min_samples_split': [2, 5, 7],
    }

    kf = KFold(
        n_splits = 10,
        shuffle = True,
        random_state = 1234,
    )

    grid_search = GridSearchCV(
        estimator = rf,
        param_grid = param_grid,
        cv = kf,
        scoring = 'f1_macro',
        n_jobs = -1,
    )

    grid_search.fit(x_train, y_train.values.ravel())

    print(f'Best Params: {grid_search.best_params_}')
    print(f'Best Score: {grid_search.best_score_:.4f}')

    best_model = grid_search.best_estimator_
    test_score = best_model.score(x_test, y_test)
    print(f'Test Score: {test_score}')

trainRandomForest(x_train, x_test, y_train, y_test)

Best Params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 175}
Best Score: 0.5660
Test Score: 0.5172413793103449


In [38]:
def trainXGB(x_train, x_test, y_train, y_test):
    xgb = XGBClassifier()

    param_grid = {
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
        'n_estimators': [20, 50, 100, 125, 150, 175, 200],
        'subsample': [0.8, 1.0]
    }

    kf = KFold(
        n_splits = 10,
        shuffle = True,
        random_state = 1234,
    )

    grid_search = GridSearchCV(
        estimator = xgb,
        param_grid = param_grid,
        cv = kf,
        scoring = 'f1_macro',
        n_jobs = -1,
    )

    grid_search.fit(x_train, y_train.values.ravel())

    print(f'Best Params: {grid_search.best_params_}')
    print(f'Best Score: {grid_search.best_score_:.4f}')

    best_model = grid_search.best_estimator_
    test_score = best_model.score(x_test, y_test)
    print(f'Test Score: {test_score}')

trainXGB(x_train, x_test, y_train, y_test)

Best Params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 150, 'subsample': 1.0}
Best Score: 0.5555
Test Score: 0.41379310344827586


Models did not perform well. I believe the high class imbalance (seen in EDA.ipynb).

## Train with 2 classes (is greater than s0)

In [39]:
binary_split = train_test_split(X, Y > 1, test_size = 0.25, random_state = 1234)

In [40]:
trainRandomForest(*binary_split)

Best Params: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 175}
Best Score: 0.7446
Test Score: 0.6551724137931034


In [41]:
trainXGB(*binary_split)

Best Params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 175, 'subsample': 0.8}
Best Score: 0.8027
Test Score: 0.6551724137931034


Training with a binary class target led to better scores, though still not great.

## Synthetic Minority Over-sampling Technique (SMOTE)

### Multi-Class

In [42]:
# Remove nan
mask = ~np.any(np.isnan(X), axis = 1)
 
x_train, x_test, y_train, y_test = train_test_split(
    X[mask], Y[mask],
    test_size = 0.3, 
    random_state = 1234
)

smote = SMOTE(random_state = 1234)
x_train, y_train = smote.fit_resample(x_train, y_train)

y_train.value_counts()

Steatosis stage
3                  33
1                  33
2                  33
0                  33
Name: count, dtype: int64

In [43]:
trainRandomForest(x_train, x_test, y_train, y_test)

Best Params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Best Score: 0.8881
Test Score: 0.6206896551724138


In [44]:
trainXGB(x_train, x_test, y_train, y_test)

Best Params: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 125, 'subsample': 0.8}
Best Score: 0.8008
Test Score: 0.6896551724137931


### Binary-Class

In [45]:
# Remove nan
mask = ~np.any(np.isnan(X), axis = 1)
 
x_train, x_test, y_train, y_test = train_test_split(
    X[mask], Y[mask] > 1,
    test_size = 0.3, 
    random_state = 1234
)

smote = SMOTE(random_state = 1234)
x_train, y_train = smote.fit_resample(x_train, y_train)

y_train.value_counts()

Steatosis stage
True               40
False              40
Name: count, dtype: int64

In [46]:
trainRandomForest(x_train, x_test, y_train, y_test)

Best Params: {'max_depth': None, 'min_samples_split': 7, 'n_estimators': 20}
Best Score: 0.8466
Test Score: 0.7241379310344828


In [47]:
trainXGB(x_train, x_test, y_train, y_test)

Best Params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 20, 'subsample': 0.8}
Best Score: 0.8294
Test Score: 0.7586206896551724


SMOTE actually helped. SMOTE with multi-class target performed better than imbalanced data with binary-class target.
SMOTE + binary-class seems to have performed the best.
It seems having more data would really be the best solution when only using the clinical metadata.

In [ ]:
# Something to keep in mind once images are used, implement patient-stratified data split